# Snack market map: principal components of 13 attributes

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Principal Component Analysis** · a worked example from the StatNex teaching guide

**The story.** A consumer panel scored 56 snacks (chips, sweets and "better-for-you" snacks at budget, standard and premium price tiers) from 1 to 10 on 13 attributes. A brand manager wants a simple map of the snack market.

**The question.** Which two or three underlying dimensions organise how consumers see snacks?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

This analysis needs `scikit-learn`. This cell installs them. If Colab asks you to restart the session, restart it and choose **Run all** again.

In [ ]:
%pip install -q "scikit-learn"

## The table

This cell saves the table `09_multivariate/principal_component_analysis.csv` (56 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Snack,Price tier,Sweetness,Saltiness,Crunchiness,Rich flavour,Healthy image,Portion size,Packaging appeal,Brand awareness,Novelty,Shareability,Kid appeal,Value for money,Repurchase intent
Sea Salt Waves,Budget,5.0,7.0,7.0,4.0,2.0,10.0,1.0,4.0,1.0,9.0,10.0,9.0,8.0
BBQ Crunchers,Budget,4.0,10.0,10.0,2.0,6.0,10.0,5.0,8.0,6.0,8.0,4.0,10.0,2.0
Cheesy Puffs,Budget,5.0,8.0,2.0,3.0,1.0,6.0,1.0,8.0,3.0,9.0,8.0,5.0,4.0
Sour Cream Ridges,Standard,2.0,8.0,7.0,5.0,6.0,7.0,8.0,4.0,8.0,5.0,6.0,1.0,2.0
Truffle Kettle Chips,Premium,2.0,8.0,3.0,6.0,6.0,5.0,9.0,8.0,3.0,3.0,4.0,4.0,4.0
Chili Lime Tortillas,Standard,7.0,9.0,8.0,7.0,6.0,5.0,3.0,5.0,7.0,10.0,5.0,3.0,2.0
Classic Pretzels,Budget,1.0,9.0,9.0,2.0,5.0,7.0,1.0,3.0,8.0,10.0,6.0,10.0,7.0
Wasabi Peas,Standard,3.0,9.0,4.0,9.0,4.0,4.0,6.0,5.0,5.0,6.0,8.0,3.0,9.0
Smoked Almonds,Premium,7.0,7.0,2.0,8.0,1.0,7.0,6.0,10.0,9.0,5.0,7.0,5.0,3.0
Butter Popcorn,Budget,2.0,10.0,10.0,1.0,7.0,10.0,3.0,3.0,2.0,10.0,4.0,9.0,2.0
Parmesan Crisps,Premium,3.0,8.0,9.0,9.0,5.0,3.0,6.0,6.0,3.0,2.0,2.0,4.0,4.0
Seaweed Snacks,Standard,3.0,10.0,9.0,2.0,2.0,6.0,4.0,5.0,1.0,7.0,5.0,7.0,4.0
Pizza Bites,Budget,2.0,10.0,8.0,4.0,5.0,9.0,2.0,9.0,3.0,9.0,7.0,8.0,2.0
Rosemary Crackers,Premium,1.0,9.0,8.0,8.0,6.0,4.0,9.0,5.0,10.0,3.0,4.0,5.0,7.0
Spicy Ramen Chips,Budget,4.0,10.0,9.0,3.0,5.0,10.0,1.0,1.0,4.0,9.0,6.0,9.0,3.0
Salted Cashews,Premium,1.0,7.0,6.0,7.0,8.0,2.0,8.0,9.0,9.0,5.0,2.0,2.0,1.0
Cheddar Crackers,Budget,2.0,9.0,4.0,1.0,5.0,9.0,2.0,2.0,1.0,7.0,6.0,10.0,6.0
Onion Rings Snack,Budget,1.0,9.0,10.0,1.0,7.0,9.0,1.0,1.0,6.0,5.0,3.0,7.0,1.0
Choco Crunch Bar,Budget,9.0,2.0,3.0,7.0,3.0,8.0,4.0,5.0,5.0,6.0,10.0,7.0,8.0
Caramel Cluster,Standard,6.0,2.0,5.0,7.0,6.0,7.0,5.0,4.0,6.0,4.0,7.0,4.0,10.0
Dark Chocolate 70%,Premium,8.0,6.0,4.0,10.0,1.0,3.0,9.0,10.0,7.0,5.0,5.0,2.0,6.0
Milk Choco Buttons,Budget,9.0,2.0,1.0,5.0,4.0,9.0,3.0,4.0,1.0,10.0,10.0,9.0,10.0
Gummy Bears,Budget,10.0,2.0,1.0,6.0,3.0,5.0,2.0,1.0,4.0,8.0,9.0,10.0,9.0
Sour Worms,Budget,8.0,4.0,1.0,2.0,4.0,4.0,5.0,9.0,2.0,4.0,10.0,6.0,7.0
Hazelnut Wafers,Standard,8.0,4.0,3.0,7.0,1.0,6.0,3.0,6.0,2.0,8.0,9.0,9.0,5.0
Salted Caramel Truffles,Premium,8.0,3.0,5.0,9.0,2.0,4.0,8.0,7.0,5.0,6.0,8.0,1.0,10.0
Double Choc Cookies,Standard,7.0,6.0,3.0,9.0,1.0,5.0,6.0,4.0,1.0,4.0,9.0,6.0,6.0
Mini Donuts,Budget,8.0,4.0,4.0,6.0,4.0,2.0,3.0,2.0,2.0,7.0,9.0,8.0,8.0
Marshmallow Twists,Budget,6.0,6.0,4.0,1.0,3.0,9.0,2.0,2.0,1.0,7.0,8.0,7.0,3.0
Matcha Kit Bar,Premium,10.0,1.0,1.0,10.0,2.0,1.0,10.0,8.0,9.0,1.0,7.0,1.0,3.0
Fudge Brownie Bites,Standard,10.0,3.0,2.0,7.0,2.0,5.0,4.0,7.0,4.0,6.0,9.0,7.0,10.0
Strawberry Licorice,Budget,9.0,1.0,1.0,6.0,4.0,6.0,1.0,1.0,4.0,8.0,7.0,6.0,8.0
Belgian Pralines,Premium,10.0,1.0,2.0,9.0,3.0,1.0,9.0,7.0,8.0,4.0,6.0,2.0,6.0
Peanut Butter Cups,Standard,7.0,1.0,1.0,6.0,4.0,4.0,5.0,9.0,7.0,2.0,7.0,4.0,7.0
Cookie Dough Bites,Standard,9.0,1.0,3.0,5.0,3.0,7.0,2.0,3.0,3.0,9.0,9.0,8.0,5.0
Honey Waffles,Premium,10.0,6.0,5.0,10.0,1.0,6.0,7.0,9.0,2.0,8.0,10.0,2.0,7.0
Jelly Beans,Budget,9.0,5.0,6.0,8.0,7.0,8.0,4.0,10.0,9.0,10.0,6.0,7.0,9.0
Choc Chip Muffin,Standard,9.0,2.0,4.0,10.0,7.0,1.0,6.0,2.0,7.0,9.0,8.0,5.0,7.0
Macaron Box,Premium,7.0,6.0,2.0,10.0,2.0,3.0,8.0,10.0,7.0,2.0,5.0,1.0,2.0
Oat Granola Bar,Standard,6.0,5.0,5.0,3.0,9.0,8.0,7.0,7.0,6.0,3.0,2.0,6.0,6.0
Protein Bar,Premium,3.0,3.0,5.0,8.0,9.0,2.0,9.0,1.0,4.0,1.0,1.0,2.0,6.0
Trail Mix,Standard,4.0,4.0,6.0,2.0,7.0,4.0,7.0,6.0,2.0,4.0,5.0,3.0,9.0
Rice Cakes,Budget,6.0,2.0,2.0,3.0,8.0,2.0,4.0,3.0,5.0,7.0,4.0,8.0,9.0
Dried Mango,Premium,5.0,7.0,7.0,5.0,8.0,3.0,7.0,8.0,7.0,6.0,1.0,5.0,4.0
Veggie Straws,Budget,1.0,5.0,8.0,1.0,10.0,10.0,4.0,1.0,6.0,6.0,2.0,9.0,3.0
Apple Chips,Standard,4.0,8.0,9.0,5.0,9.0,7.0,6.0,6.0,10.0,4.0,1.0,8.0,4.0
Greek Yogurt Bites,Premium,4.0,6.0,6.0,1.0,7.0,3.0,10.0,10.0,9.0,1.0,3.0,2.0,9.0
Chickpea Crisps,Premium,3.0,3.0,6.0,4.0,10.0,2.0,8.0,6.0,10.0,1.0,1.0,3.0,1.0
Kale Chips,Premium,1.0,7.0,9.0,4.0,8.0,8.0,7.0,4.0,9.0,1.0,1.0,4.0,5.0
Fruit Leather,Budget,2.0,5.0,8.0,2.0,10.0,9.0,5.0,9.0,8.0,3.0,3.0,6.0,5.0
Almond Butter Bites,Premium,5.0,5.0,10.0,4.0,10.0,1.0,10.0,7.0,10.0,2.0,2.0,4.0,8.0
Banana Chips,Budget,7.0,4.0,10.0,3.0,9.0,8.0,2.0,2.0,5.0,7.0,4.0,10.0,10.0
Seed Crackers,Premium,4.0,3.0,7.0,9.0,10.0,2.0,10.0,2.0,10.0,1.0,1.0,1.0,1.0
Coconut Clusters,Premium,5.0,4.0,6.0,8.0,9.0,1.0,10.0,7.0,6.0,2.0,3.0,3.0,1.0
Berry Oat Bites,Standard,6.0,1.0,7.0,4.0,9.0,6.0,7.0,3.0,4.0,2.0,3.0,6.0,1.0
Energy Balls,Premium,6.0,7.0,7.0,6.0,8.0,1.0,9.0,6.0,8.0,3.0,2.0,1.0,5.0
""")

## Helper: redundant_variables

A step that no library does, written out. In the code download it is the file `redundant_variables.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Which variables are exact linear combinations of the ones listed before them.

A variable that is a weighted sum of others adds no information and makes the
correlation matrix singular. No library names them, so it is written out: each
column, centred and scaled to unit length, is regressed on the columns kept so far,
and a residual sum of squares of 1e-10 or less marks it redundant. A constant column
is redundant too. The later member of a redundant pair is the one reported, so the
variable listed first survives.
"""

import numpy as np


def redundant_variables(x, names, tolerance=1e-10):
    centred = x - x.mean(axis=0)
    length = np.linalg.norm(centred, axis=0)
    length[length == 0] = 1.0
    centred = centred / length
    redundant, kept = [], []
    for j, name in enumerate(names):
        residual = centred[:, j]
        if kept:
            basis = centred[:, kept]
            residual = residual - basis @ np.linalg.lstsq(basis, residual, rcond=None)[0]
        if residual @ residual <= tolerance or x[:, j].max() == x[:, j].min():
            redundant.append(name)
        else:
            kept.append(j)
    return redundant

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.decomposition import PCA
# redundant_variables: defined in the helper cell redundant_variables above (redundant_variables.py in the code download)

pd.set_option("display.width", 120)

## The data

In [ ]:
# Text and blank cells count as missing, and a case missing any variable is left out.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Sweetness", "Saltiness", "Crunchiness", "Rich flavour", "Healthy image", "Portion size", "Packaging appeal", "Brand awareness", "Novelty", "Shareability", "Kid appeal", "Value for money", "Repurchase intent"]
label_column = ["Snack"]
numbers = data[variables].apply(pd.to_numeric, errors="coerce")
complete = numbers.notna().all(axis=1)
x = numbers[complete].to_numpy(float)
n, p = x.shape
cases = (data.loc[complete, label_column[0]].astype(str).tolist() if label_column
         else [str(i + 1) for i in range(n)])

summary = pd.DataFrame({"n": [n, len(data) - n, len(data)]},
                       index=["Valid cases", "Incomplete cases", "Total"])
summary["share"] = summary["n"] / len(data)
print(summary)

## Descriptive statistics

In [ ]:
# Variance and standard deviation divide by n - 1.
describe = pd.DataFrame(
    {"n": n, "mean": x.mean(axis=0), "variance": x.var(axis=0, ddof=1),
     "SD": x.std(axis=0, ddof=1), "min": x.min(axis=0), "max": x.max(axis=0)},
    index=variables,
)
print()
print(describe)

## Correlation matrix

In [ ]:
# The components are found from the correlation matrix, so every variable counts
# equally whatever its units. That is the same as giving PCA below the standardised
# values: each value minus its variable's mean, divided by its SD (with n - 1).
analysed = np.corrcoef(x, rowvar=False)
prepared = (x - x.mean(axis=0)) / x.std(axis=0, ddof=1)
print()
print(pd.DataFrame(analysed, index=variables, columns=variables).round(4))
print("\nLinear combinations of other variables:", redundant_variables(x, variables) or "none")

## Eigenvalues

In [ ]:
# scikit-learn's PCA finds the components by a singular value decomposition of the
# centred data, which gives exactly the eigenvalues and eigenvectors of the matrix
# above, largest first. Each eigenvalue is the variance of one component
# (explained_variance_, which divides by n - 1, as the matrix does); together they add
# up to the total variance of that matrix -- on the correlation matrix, the number of
# variables, so a component with an eigenvalue above 1 carries more than one
# variable's worth. The share is each eigenvalue over that total.
pca = PCA(svd_solver="full").fit(prepared)
eigenvalues = pca.explained_variance_
vectors = pca.components_.T
share = eigenvalues / eigenvalues.sum()
eigen = pd.DataFrame({"eigenvalue": eigenvalues, "share": share, "cumulative": share.cumsum()},
                     index=range(1, p + 1))
print()
print(eigen)

## Eigenvectors and loadings

In [ ]:
# An eigenvector (one column of weights per component) and its negative describe the
# same component, so each one is flipped to make its elements sum to a positive
# number; packages pick the sign arbitrarily. A loading is the correlation between a
# variable and a component: the eigenvector times the square root of its eigenvalue.
# Read the loadings, not the eigenvectors, to name a component: the variables with
# the largest loadings, positive and negative, are what it contrasts.
signs = np.where(vectors.sum(axis=0) < 0, -1.0, 1.0)
vectors = vectors * signs
keep = min(8, p)
components = [f"PC{j + 1}" for j in range(keep)]
eigenvectors = pd.DataFrame(vectors[:, :keep], index=variables, columns=components)
loadings = eigenvectors * np.sqrt(eigenvalues[:keep])
print()
print(eigenvectors.round(4))
print()
print(loadings.round(4))

## Component scores

In [ ]:
# Each case's standardised values (SD with n - 1) times the eigenvectors: PCA's
# transform(), with the same sign flips. A case's scores place it on the map; a score
# of 0 is the average case.
scores = pd.DataFrame(pca.transform(prepared)[:, :keep] * signs[:keep], index=cases,
                      columns=components)
print()
print(scores.round(4))

## Sampling adequacy (KMO) and Bartlett's test of sphericity

In [ ]:
# Are the variables related enough to be worth summarising? Both checks use the
# correlation matrix, whichever matrix was analysed, and both need its inverse, so
# neither is computed when a variable is a linear combination of the others.
# Neither SciPy nor statsmodels has them, so they are written out as R's psych
# package computes them.
# KMO (psych::KMO): the squared correlations over themselves plus the squared partial
# correlations (the links left after holding every other variable fixed, read off
# the inverse), off the diagonal -- per variable down its column, and overall. Near 1,
# the links are shared by the whole set; low, they are private to a few pairs (Kaiser
# 1974: 0.9 marvellous, 0.8 meritorious, 0.7 middling, 0.6 mediocre, 0.5 miserable,
# below that unacceptable -- a rule of thumb, not a test).
# Bartlett's test of sphericity (psych::cortest.bartlett): H0 is that the population
# correlation matrix is the identity, every pair uncorrelated; -(n - 1 - (2p + 5)/6)
# log|R| is chi-square on p(p - 1)/2 df if the data are multivariate normal. A small
# P says only that some correlations are not zero.
if not redundant_variables(x, variables):
    correlation = np.corrcoef(x, rowvar=False)
    inverse = np.linalg.inv(correlation)
    partial = inverse / np.sqrt(np.outer(np.diag(inverse), np.diag(inverse)))
    r2, a2 = correlation ** 2, partial ** 2
    np.fill_diagonal(r2, 0)
    np.fill_diagonal(a2, 0)
    kmo = pd.Series([*(r2.sum(axis=0) / (r2.sum(axis=0) + a2.sum(axis=0))),
                     r2.sum() / (r2.sum() + a2.sum())], index=[*variables, "overall"])
    bartlett_chi2 = -(n - 1 - (2 * p + 5) / 6) * np.linalg.slogdet(correlation)[1]
    bartlett_df = p * (p - 1) // 2
    bartlett = [bartlett_chi2, bartlett_df, stats.chi2.sf(bartlett_chi2, bartlett_df)]
    print()
    print(kmo.round(4))
    print(f"Bartlett's test of sphericity: chi-square {bartlett_chi2:.4f}, df {bartlett_df}, "
          f"P {bartlett[2]:.4g}")

## What the result says

Principal component analysis (PCA) squeezes many correlated measures into a few new axes, called components, that keep as much of the variation as possible.

The eigenvalue table shows how much each component keeps. The first holds 33% and the second 29%, so two components carry 62% of the differences between the 56 snacks. The third adds only 11%.

The component loadings, the correlations between each attribute and a component, give the axes their meaning. Component 1 sets shareability (0.82) and value for money (0.77) against packaging appeal (-0.90) and novelty (-0.73): mass-market against premium. Component 2 sets crunchiness (0.83) and saltiness (0.66) against sweetness (-0.83): savoury against sweet. So two dimensions are enough for a simple two-axis map of the snack market.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.